# Assignment: Cleaned Company Employee Dataset (Day 11)

In [ ]:
import pandas as pd
import numpy as np

# Load raw dataset
raw_file = 'Day11_Messy_Company_Employee_Dataset.csv'
df = pd.read_csv(raw_file)

print('--- Dataset Shape ---')
print(df.shape)

print('\n--- Dataset Overview ---')
print(df.info())

df.head()

## Step 1: Data Inspection & Quantifying Issues

In [ ]:
# Missing values count per column
print('--- Missing Values Count ---')
print(df.isnull().sum())
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')

# Duplicate records count
print(f'\nTotal Duplicate Rows: {df.duplicated().sum()}')

# Unique entries in categorical columns to identify casing/whitespace issues
cat_cols = ['Department', 'Gender', 'City', 'Work_Mode']
for col in cat_cols:
    print(f'\nUnique values in {col}:')
    print(df[col].unique())

## Step 2: Data Cleaning Operations

In [ ]:
df_clean = df.copy()

# 1. Deduplication
df_clean = df_clean.drop_duplicates()

# 2. Text Normalization & Whitespace Removal
for col in ['Department', 'Gender', 'City', 'Work_Mode']:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
    df_clean[col] = df_clean[col].replace('Nan', np.nan)

df_clean['Gender'] = df_clean['Gender'].str.capitalize()
df_clean['Work_Mode'] = df_clean['Work_Mode'].str.capitalize()

# 3. Missing Value Imputations
# Impute Department based on Job_Title mode mapping
job_dept_map = df_clean.groupby('Job_Title')['Department'].agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
df_clean['Department'] = df_clean['Department'].fillna(df_clean['Job_Title'].map(job_dept_map))

# Impute Annual_Salary and Experience_Years by Job_Title median
df_clean['Annual_Salary'] = df_clean['Annual_Salary'].fillna(
    df_clean.groupby('Job_Title')['Annual_Salary'].transform('median')
).fillna(df_clean['Annual_Salary'].median())

df_clean['Experience_Years'] = df_clean['Experience_Years'].fillna(
    df_clean.groupby('Job_Title')['Experience_Years'].transform('median')
).fillna(df_clean['Experience_Years'].median())

# Impute Age using global median
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Impute remaining categoricals and discrete values using Mode
df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].mode()[0])
df_clean['City'] = df_clean['City'].fillna(df_clean['City'].mode()[0])
df_clean['Performance_Score'] = df_clean['Performance_Score'].fillna(df_clean['Performance_Score'].mode()[0])
df_clean['Work_Mode'] = df_clean['Work_Mode'].fillna(df_clean['Work_Mode'].mode()[0])

# 4. Data Type Conversion
df_clean['Joining_Date'] = pd.to_datetime(df_clean['Joining_Date'])
df_clean['Age'] = df_clean['Age'].astype(int)
df_clean['Performance_Score'] = df_clean['Performance_Score'].astype(int)

## Step 3: Verification & Condition Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Total Rows', 'Duplicate Rows', 'Total Missing Values', 'Joining_Date Dtype', 'Age Dtype', 'Performance_Score Dtype'],
    'Before Cleaning': [len(df), df.duplicated().sum(), df.isnull().sum().sum(), str(df['Joining_Date'].dtype), str(df['Age'].dtype), str(df['Performance_Score'].dtype)],
    'After Cleaning': [len(df_clean), df_clean.duplicated().sum(), df_clean.isnull().sum().sum(), str(df_clean['Joining_Date'].dtype), str(df_clean['Age'].dtype), str(df_clean['Performance_Score'].dtype)]
})

print('--- Dataset Condition Comparison ---')
print(comparison)

print('\n--- Cleaned DataFrame Info ---')
df_clean.info()

## Step 4: Export Cleaned Dataset

In [ ]:
output_path = 'Cleaned_Company_Employee_Dataset.csv'
df_clean.to_csv(output_path, index=False)
print(f'Cleaned dataset successfully saved as {output_path}')